In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_excel("../data/Telco_Customer_Churn.xlsx")

In [20]:
# Lower each column name
df.columns = df.columns.str.lower()

# Drop irrelevant information
df = df.drop(columns=["customerid", "count", "lat long", "latitude", "longitude", "churn label", "churn score", "churn reason", "city", "zip code", "country", "state"], errors = "ignore")

In [21]:
print(f"Amount of total charges that are null: {df["total charges"].isnull().sum()}")

df["total charges"] = pd.to_numeric(df["total charges"], errors = "coerce") # convert invalid to NaN
df["total charges"] = df["total charges"].fillna(0) # convert NaN to to 0



Amount of total charges that are null: 0


In [22]:
columns_to_change_to_binary = ["gender", "senior citizen", "partner", "dependents", "phone service", "multiple lines", "online security", "online backup", "device protection", "tech support", "streaming tv", "streaming movies", "paperless billing"]

for column in columns_to_change_to_binary:
    df[column] = df[column].map({
        "Female":0, 
        "Male":1,
        "No":0,
        "No phone service":0,
        "No internet service":0,
        "Yes":1,
        })



In [ ]:
subscription_services = ["phone service", "multiple lines", "online security", "online backup", "device protection", "tech support", "streaming tv", "streaming movies"]

df["number of subscriptions"] = df[subscription_services].apply(lambda row: (row == 1).sum(), axis=1)

df["average charge per subscription"] = df.apply(lambda row: row["monthly charges"] / row["number of subscriptions"] if row["number of subscriptions"] > 0 else 0,
                                                 axis = 1)

print(f"Maximum number of months a customer has been subscribed: {df["tenure months"].max()}\n")

df["tenure month type"] = pd.cut( # group tenure months into bins
    df["tenure months"],
    bins = [0, 12, 24, 48, 72],
    labels = ["0 to 12", "12 to 24", "24 to 48", "48 to 72"] # 0<=m<12, 12<=m<24, 24<=m<48, 48<=m<=72
)

print(df[["number of subscriptions", "average charge per subscription", "tenure month type"]].head(10).to_string(index=False))


Maximum number of months a customer has been subscribed: 72

 number of subscriptions  average charge per subscription tenure month type
                       3                        17.950000           0<=m<12
                       1                        70.700000           0<=m<12
                       5                        19.930000           0<=m<12
                       6                        17.466667           24<m<48
                       6                        17.283333          48<m<=72
                       3                        18.400000           0<=m<12
                       2                        19.825000           0<=m<12
                       1                        20.150000           0<=m<12
                       5                        19.870000           24<m<48
                       1                        30.200000           0<=m<12


In [24]:
df["contract"] = df["contract"].map({
    "Month-to-month":0,
    "One year":1,
    "Two year":2
    })



In [25]:
# One hot encoding categorical variables
df = pd.get_dummies(df, columns=["internet service", "payment method", "tenure month type"]) # bool type

columns_with_bool_type = df.select_dtypes(include="bool").columns
df[columns_with_bool_type] = df[columns_with_bool_type].astype(int) # convert to 0/1

In [26]:
churn_counts = df["churn value"].value_counts()
churn_percentage = df["churn value"].value_counts(normalize=True) * 100 # percent version

print("Class distribution:")
for label, count, percent in zip(churn_counts.index, churn_counts, churn_percentage): # guaranteed same length
    print(f"{'Churned' if label == 1 else 'Retained'}: {count} ({percent:.1f}%)")

Class distribution:
Retained: 5174 (73.5%)
Churned: 1869 (26.5%)


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns="churn value"), # new df, not modify
    df["churn value"],
    test_size = 0.2,
    random_state = 42, # the answer
    stratify=df["churn value"] # preserves 73.5%, 26.5% split
)

X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)
